In [0]:
%run "../libs/libs"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import current_timestamp

In [0]:
df_ordertracking = spark.read.format("delta").table("dev.tracking_br.ordertracking")

In [0]:
from pyspark.sql.functions import col, to_date
df_ordertracking = df_ordertracking.select(col("trackingid"),
                               col("shipmentno"),
                               col("orderstatus"),
                               col("createdon").alias("trackingtime"))

                               

In [0]:
from pyspark.sql.functions import current_timestamp
df_ordertracking = convert_to_int(df_ordertracking, "trackingid")
df_ordertracking = df_ordertracking.withColumn("ingestion_time",current_timestamp())
df_ordertracking = convert_to_timestamp(df_ordertracking, "trackingtime")

In [0]:
df_ordertracking.createOrReplaceTempView("temp_ordertracking")

In [0]:
%sql
MERGE INTO dev.tracking_sl.ordertracking tgt 
USING ( 
  SELECT DISTINCT * 
  FROM temp_ordertracking 
) src 
ON tgt.trackingid = src.trackingid 
WHEN NOT MATCHED THEN 
  INSERT *